# 02 LLM 连通性体检

**测什么**: 用项目实际配置的两个模型(`get_executor_llm` = flash / `get_planner_llm` = pro),
对图真正依赖的三条调用形态逐一实测:

| 调用形态 | 图内使用点 |
|---|---|
| 普通对话 `ainvoke` | `finalize`(组装答复) |
| 工具调用 `bind_tools` | `executor`(为步骤生成 tool_calls) |
| 结构化输出 `with_structured_output` | `planner` / `replanner`(计划)、`replan_check` / `element_assess` / `mid_clarify`(判定) |

后两条是 v2 架构的骨架: 结构化输出失败时 planner 会退到默认 4 步计划、replan_check 会退到规则判断,
图仍能跑但智能程度下降, 故必须单列诊断。

**判定**: 每个 (模型 × 调用形态) 组合单独判定。基础连通不过 → 全册无意义;
某个形态失败 → 只有走该形态的节点降级, 会明确列出受影响的节点。

**前置**: 01 册的 `DEEPSEEK_API_KEY` 已填。本册真实调用 API, 约 6 次小请求。


In [1]:
import asyncio, os, sys
from pathlib import Path

for cand in (Path.cwd(), *Path.cwd().parents):
    if (cand / "nbkit.py").is_file():
        NB_DIR = cand
        break
    if (cand / "tests_ipynb" / "nbkit.py").is_file():
        NB_DIR = cand / "tests_ipynb"
        break
else:
    raise RuntimeError("未找到 nbkit.py")

sys.path.insert(0, str(NB_DIR))

from nbkit import Checks, bootstrap

ROOT = bootstrap()
checks = Checks("02 LLM 连通性体检")

print("解释器  :", sys.executable)
print("仓库根  :", ROOT)
print("HF_HOME :", os.getenv("HF_HOME", "(未设置)"))


解释器  : F:\Anaconda_env\lawApp_langGraph\python.exe
仓库根  : E:\LangChain_LawAgent-main
HF_HOME : E:\huggingface_cache


In [2]:
import time
from lawApp_LangGraph.config import settings as s

RUN_LLM_CALLS = True
key_ok = bool((s.deepseek_api_key or "").strip())
RUN = RUN_LLM_CALLS and key_ok

if not key_ok:
    checks.fail("LLM 凭据", "DEEPSEEK_API_KEY 为空 → 本册全部跳过")
print("pro 模型    :", s.deepseek_pro_model)
print("flash 模型  :", s.deepseek_flash_model)
print("Base URL    :", s.deepseek_base_url)
print("是否实调 API:", RUN)

from lawApp_LangGraph.LangGraph_lawApp import (
    ReplanCheckSchema,
    get_executor_llm,
    get_planner_llm,
)

pro 模型    : deepseek-reasoner
flash 模型  : deepseek-chat
Base URL    : https://api.deepseek.com
是否实调 API: True


F:\Anaconda_env\lawApp_langGraph\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 1. 基础连通性(普通对话)

In [3]:
async def plain(llm_getter, label, timeout=90):
    """发一次最小对话, 返回 (是否成功, 说明文本)。"""
    t0 = time.time()
    try:
        msg = await asyncio.wait_for(llm_getter().ainvoke("只回复两个字: 可用"), timeout)
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e)[:160]}"
    text = str(getattr(msg, "content", "")).strip()
    return bool(text), f"{time.time() - t0:.1f}s | {text[:40]!r}"

if RUN:
    for label, getter in (("executor(flash)", get_executor_llm), ("planner(pro)", get_planner_llm)):
        ok, detail = await plain(getter, label)
        checks.expect(ok, f"普通对话: {label}", ok_detail=detail, fail_detail=detail)
else:
    checks.skip("普通对话", "无密钥")

[PASS] 普通对话: executor(flash) | 5.1s | '可用'


[PASS] 普通对话: planner(pro) | 0.7s | '可用'


## 2. 能力矩阵: 工具调用 与 结构化输出

In [4]:
# 代码实际使用的结构化输出路径(T1 起四节点 json_mode; T4 后 planner/replanner 手工流式)
_SRC = (ROOT / "lawApp_LangGraph" / "LangGraph_lawApp.py").read_text(encoding="utf-8")
_FLASH_JSON_MODE = all(
    f'with_structured_output({schema}, method="json_mode")' in _SRC
    for schema in ("RiskSchema", "ElementAssessmentSchema", "ReplanCheckSchema", "MidClarifySchema")
)
_PRO_LEGACY_SCHEMA = "with_structured_output(PlanSchema)" in _SRC

async def tool_calling(llm_getter, timeout=90):
    """测 bind_tools 是否能产出 tool_calls(executor 依赖)。"""
    from lawApp_LangGraph.tools.db_tools import fetch_laws

    t0 = time.time()
    try:
        msg = await asyncio.wait_for(
            llm_getter()
            .bind_tools([fetch_laws])
            .ainvoke("用工具查一下: 离婚财产分割 的法律条文。必须调用工具。"),
            timeout,
        )
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e)[:160]}"
    calls = getattr(msg, "tool_calls", None) or []
    return bool(calls), f"{time.time() - t0:.1f}s | tool_calls={len(calls)}"


async def structured(llm_getter, method, timeout=120):
    """测 with_structured_output 能否返回 Pydantic 实例(planner/replan_check 依赖)。

    json_mode 下 LangChain 不会把 schema 传给服务端, 必须在提示词里声明字段,
    否则模型只能自造字段名而必然解析失败——那种失败是测法问题, 不是栈的问题。
    """
    t0 = time.time()
    prompt = (
        "判断这条咨询的信息是否不足以生成回答。只输出 JSON 对象, 字段名必须与下面完全一致:\n"
        '{"needs_replan": true 或 false, "reason": "不超过50字的依据", '
        '"insufficient_reason": "vague|not_found|error|none 四选一"}\n'
        "咨询内容: 我打算离婚, 房子婚后买的。"
    )
    try:
        llm = llm_getter()
        chain = (
            llm.with_structured_output(ReplanCheckSchema, method=method)
            if method
            else llm.with_structured_output(ReplanCheckSchema)
        )
        verdict = await asyncio.wait_for(chain.ainvoke(prompt), timeout)
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e)[:160]}"
    ok = isinstance(verdict, ReplanCheckSchema)
    return ok, f"{time.time() - t0:.1f}s | needs_replan={getattr(verdict, 'needs_replan', '?')}"


MATRIX = [
    ("flash", get_executor_llm, "tool_calling", tool_calling, None),
    ("flash", get_executor_llm, "structured(json_schema)", structured, None),
    ("flash", get_executor_llm, "structured(json_mode)", structured, "json_mode"),
    ("pro", get_planner_llm, "tool_calling", tool_calling, None),
    ("pro", get_planner_llm, "structured(json_schema)", structured, None),
    ("pro", get_planner_llm, "structured(json_mode)", structured, "json_mode"),
]

capability = {}
if RUN:
    for label, getter, kind, fn, method in MATRIX:
        if kind == "tool_calling":
            ok, detail = await fn(getter)
        else:
            ok, detail = await fn(getter, method)
        capability[(label, kind)] = ok
        # json_schema 不可用是 DeepSeek 的已知栈事实; 代码已切换 json_mode 的模型,
        # 该路径不再被图使用 → 降级为 SKIP, 不再计为活缺陷
        if kind == "structured(json_schema)" and not ok:
            switched = _FLASH_JSON_MODE if label == "flash" else not _PRO_LEGACY_SCHEMA
            if switched:
                checks.skip(
                    f"{label} × {kind}",
                    "DeepSeek 拒绝 json_schema(HTTP 400); 该模型节点已改 json_mode, 代码不再使用此路径",
                )
                continue
        checks.expect(
            ok,
            f"{label} × {kind}",
            ok_detail=detail,
            fail_detail=detail,
        )
else:
    for label, _getter, kind, _fn, _method in MATRIX:
        checks.skip(f"{label} × {kind}", "无密钥")


[PASS] flash × tool_calling | 1.0s | tool_calls=1
[SKIP] flash × structured(json_schema) | DeepSeek 拒绝 json_schema(HTTP 400); 该模型节点已改 json_mode, 代码不再使用此路径


[PASS] flash × structured(json_mode) | 0.9s | needs_replan=False


[PASS] pro × tool_calling | 0.9s | tool_calls=1


[FAIL] pro × structured(json_schema) | BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request


[PASS] pro × structured(json_mode) | 1.5s | needs_replan=True


## 3. 受影响的图节点与补救路径

In [5]:
NODE_DEPS = [
    ("planner / replanner", "pro"),
    ("replan_check / element_assess / mid_clarify", "flash"),
]
if RUN:
    for nodes, model in NODE_DEPS:
        # 该模型节点在代码里已切换到 json_mode → 可用性以 json_mode 实测为准
        switched = _FLASH_JSON_MODE if model == "flash" else not _PRO_LEGACY_SCHEMA
        if switched:
            checks.expect(
                capability.get((model, "structured(json_mode)")),
                f"节点可用: {nodes}",
                ok_detail=f"{model} × json_mode 实测可用, 代码已切换",
                fail_detail="代码已改 json_mode 但实测失败 → 检查提示词尾部的字段声明",
            )
        elif capability.get((model, "structured(json_schema)")):
            checks.ok(f"节点可用: {nodes}", f"{model} × structured(json_schema) 原生可用")
        elif capability.get((model, "structured(json_mode)")):
            checks.fail(
                f"节点可用: {nodes}",
                f"代码现用的 json_schema 不可用 → 该节点走兜底(默认 4 步计划 / 规则判断); "
                f"补救: 改 method='json_mode' 并在提示词内声明字段(json_mode 实测可用)",
            )
        else:
            checks.fail(
                f"节点可用: {nodes}",
                f"{model} 两条结构化输出路径都不可用 → 只能走兜底, 需换模型或改回「剥栅栏 + json.loads」",
            )
    checks.expect(
        capability.get(("flash", "tool_calling")),
        "节点可用: executor",
        ok_detail="flash × tool_calling 可用",
        fail_detail="flash 无法生成 tool_calls → 全流程无法执行工具",
    )
    plain_ok = capability.get(("flash", "plain"), True)
    checks.ok("节点可用: finalize", "flash 普通对话可用" if plain_ok else "见上一节")
else:
    checks.skip("受影响的图节点", "无密钥")

[FAIL] 节点可用: planner / replanner | 代码现用的 json_schema 不可用 → 该节点走兜底(默认 4 步计划 / 规则判断); 补救: 改 method='json_mode' 并在提示词内声明字段(json_mode 实测可用)
[PASS] 节点可用: replan_check / element_assess / mid_clarify | flash × json_mode 实测可用, 代码已切换
[PASS] 节点可用: executor | flash × tool_calling 可用
[PASS] 节点可用: finalize | flash 普通对话可用


## 汇总

In [6]:
print(checks.report())


02 LLM 连通性体检 — 汇总
✓ 普通对话: executor(flash)                             PASS  5.1s | '可用'
✓ 普通对话: planner(pro)                                PASS  0.7s | '可用'
✓ flash × tool_calling                                  PASS  1.0s | tool_calls=1
- flash × structured(json_schema)                       SKIP  DeepSeek 拒绝 json_schema(HTTP 400); 该模型节点已改 json_mode, 代码不再使用此路径
✓ flash × structured(json_mode)                         PASS  0.9s | needs_replan=False
✓ pro × tool_calling                                    PASS  0.9s | tool_calls=1
✗ pro × structured(json_schema)                         FAIL  BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request
✓ pro × structured(json_mode)                           PASS  1.5s | needs_replan=True
✗ 节点可用: planner / replanner                         FAIL  代码现用的 json_schema 不可用 → 该节点走兜底(默认 4 步计划 / 规则判断); 补救: 改 method='json_mode' 并在提示